# Construction du Pipeline de Machine Learning

### Initialisation de Spark et Chargement des Données

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

df = spark.read.csv("../data/db_final.csv", header=True, inferSchema=True)
df.show(5)


+-----------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------+
|CreditScore|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited| GeographyVec|
+-----------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------+
|        619|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|(3,[0],[1.0])|
|        608|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|(3,[2],[1.0])|
|        502|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|(3,[0],[1.0])|
|        699|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|     0|(3,[0],[1.0])|
|        850|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|(3,[2],[1.0])|
+-----------+------+---+------+---------+-------------+-

### Équilibrage des Données par Sous-échantillonnage

In [2]:
from pyspark.sql.functions import col

clients_actifs = df.filter(col("Exited") == 0)

clients_inactifs = df.filter(col("Exited") == 1)

print("Clients actifs :", clients_actifs.count())
print("Clients inactifs :", clients_inactifs.count())


Clients actifs : 7963
Clients inactifs : 2037


In [3]:
ratio = clients_inactifs.count() / clients_actifs.count()

clients_actifs_sampled = clients_actifs.sample(withReplacement=False, fraction=ratio, seed=42)

df_balanced = clients_inactifs.union(clients_actifs_sampled)

print("Clients actifs après sous-échantillonnage :", clients_actifs_sampled.count())
print("Clients inactifs :", clients_inactifs.count())


Clients actifs après sous-échantillonnage : 2119
Clients inactifs : 2037


### Sélection des Features

In [4]:
features = []
for c in df_balanced.columns:
    if c != "Exited":
        features.append(c)

print("Features utilisées :", features)

Features utilisées : ['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'GeographyVec']


### Configuration du Préprocessing (VectorAssembler et StandardScaler)

In [5]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

assembler = VectorAssembler(inputCols=features, outputCol="features_vector")

scaler = StandardScaler(inputCol="features_vector", outputCol="features_scaled", withMean=False, withStd=True)


### Configuration du Modèle Random Forest

In [6]:
from pyspark.ml.classification import RandomForestClassifier

rf_model = RandomForestClassifier(labelCol="Exited", featuresCol="features_scaled", numTrees=50, seed=42)


### Construction du Pipeline de Machine Learning

In [7]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[assembler, scaler, rf_model])

### Division des Données (Train/Test Split)

In [8]:
train, test = df_balanced.randomSplit([0.8, 0.2], seed=42)

print("Taille du train :", train.count())
print("Taille du test :", test.count())

Taille du train : 3375
Taille du test : 781


### Sauvegarde du Pipeline Non-Entraîné

In [10]:
save_path = "../models/pipeline_rf_untrained"
pipeline.write().overwrite().save(save_path)
